# 01 — Aggregate Metadata

Builds the spine table: every DragonForce studio album + track, pulled from MusicBrainz.
Everything downstream (lyrics, key-change annotation, whatever audio data we salvage) joins to this table on `album` + `track_title`.

Run cells top to bottom the first time. After that, feel free to jump around — that's the whole point of notebooks.

In [4]:
import sys
sys.path.append("..")

import pandas as pd
from src.musicbrainz_client import get_release_groups, get_release_tracks

In [5]:
# Pull the album list. First run hits the MusicBrainz API and caches the raw
# JSON to data/raw/ (~1 request/sec, so give it a moment). Every run after that
# reads from disk — instant, and works with the wifi off. Pass refresh=True to
# force a fresh pull when you actually want updated data.
albums = get_release_groups()
albums_df = pd.DataFrame(albums)
albums_df[["title", "first-release-date", "primary-type"]].sort_values("first-release-date")

,title,first-release-date,primary-type
5,Valley of the Damned,2003-01-27,Album
8,Sonic Firestorm,2004-03-24,Album
7,Inhuman Rampage,2005-12-28,Album
4,Ultra Beatdown,2008-08-20,Album
2,Twilight Dementia,2010-09-08,Album
6,The Power Within,2012-04-11,Album
0,Maximum Overload,2014-08-08,Album
9,In the Line of Fire… Larger Than Live,2014-08-28,Album
12,Killer Elite,2016-04-13,Album
1,Reaching Into Infinity,2017-05-17,Album


In [6]:
# Filter to studio albums only (drop live albums, compilations, remix albums)
# before pulling tracks — check this list against the README's known discography
# and adjust the filter if MusicBrainz's "secondary-types" tagging is inconsistent.
studio_albums_df = albums_df[albums_df["secondary-types"].apply(lambda x: len(x) == 0)]
studio_albums_df[["title", "first-release-date"]]

,title,first-release-date
0,Maximum Overload,2014-08-08
1,Reaching Into Infinity,2017-05-17
3,Extreme Power Metal,2019-09-25
4,Ultra Beatdown,2008-08-20
5,Valley of the Damned,2003-01-27
6,The Power Within,2012-04-11
7,Inhuman Rampage,2005-12-28
8,Sonic Firestorm,2004-03-24
10,Warp Speed Warriors,2024-03-13


In [7]:
# NOTE: get_release_tracks() needs a *release* MBID (a specific pressing),
# not a release-group MBID (the abstract "album" entity). You'll need to
# pick one release per album — e.g. via album["releases"][0]["id"] once you
# fetch releases for each release-group, or look them up manually on
# musicbrainz.org for the canonical/original pressing of each album.
#
# Left as a TODO since which pressing counts as canonical is a judgment
# call worth making deliberately rather than defaulting to "whatever's first".

In [8]:
# Once track-level data is joined in, save the spine table:
# studio_albums_df.to_csv("../data/processed/albums.csv", index=False)